# Persistência de Dados com EC2 + EBS usando Boto3

## Objetivo

Este projeto tem como objetivo demonstrar, na prática, o funcionamento de instâncias EC2 e volumes EBS na AWS utilizando a biblioteca Python Boto3.

O foco principal é compreender o ciclo de vida de uma instância EC2 e a persistência de dados provida pelo Elastic Block Store (EBS), mostrando que os dados armazenados em um volume EBS continuam existindo mesmo após a destruição da máquina virtual à qual estavam conectados anteriormente.

---

## Conceitos abordados

Durante o projeto serão utilizados os seguintes conceitos de computação em nuvem:

* Instâncias EC2
* Volumes EBS
* Ciclo de vida de máquinas virtuais
* Persistência de armazenamento
* Operações de attach/detach de volumes
* Automação com Boto3
* Provisionamento programático de infraestrutura

---

## Fluxo do experimento

O experimento será executado nas seguintes etapas:

1. Criação de uma instância EC2
2. Criação de um volume EBS
3. Associação do volume à instância
4. Criação de um arquivo dentro do volume
5. Encerramento da instância EC2
6. Criação de uma nova instância
7. Reconexão do mesmo volume EBS
8. Verificação da persistência dos dados

---

## Resultado esperado

Ao final do experimento, o arquivo criado inicialmente deverá continuar presente no volume EBS mesmo após a destruição da primeira instância EC2.

Isso demonstra uma das principais características do EBS:

> O armazenamento é persistente e independente do ciclo de vida da instância virtual.

---

## Tecnologias utilizadas

* Python
* Boto3
* AWS EC2
* AWS EBS
* SSH/Linux

---

## Motivação

Em ambientes de computação em nuvem, máquinas virtuais podem ser criadas e destruídas dinamicamente. Dessa forma, separar computação (EC2) de armazenamento persistente (EBS) é essencial para garantir durabilidade dos dados e flexibilidade operacional.

Este notebook busca ilustrar esse comportamento de forma simples e prática.


In [89]:
import boto3
from botocore.exceptions import ClientError
import os
from pathlib import Path
import sys
import paramiko


root = Path().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from utils.aws_credentials import configure_local_aws_credentials

configure_local_aws_credentials()

In [90]:
ec2_resource = boto3.resource('ec2')
for instance in ec2_resource.instances.all():
    print(instance.id, instance.state)

i-0b3c7f669812cae9c {'Code': 16, 'Name': 'running'}
i-0b80abb193481eda4 {'Code': 48, 'Name': 'terminated'}
i-0bc2c71805963f06f {'Code': 16, 'Name': 'running'}
i-041340aaacd83b606 {'Code': 48, 'Name': 'terminated'}
i-0de1b0f4188340ee2 {'Code': 48, 'Name': 'terminated'}


## Reunindo informações para criação da instância

In [91]:
# Vou pegar o id do security group 'default' para usar na criação da instância
iam = boto3.client('iam')

security_group_id = None
try:
    security_groups = ec2_resource.security_groups.filter(GroupNames=['default'])
    for group in security_groups:
        print(group.id)
        security_group_id = group.id
except StopIteration:
    print("Security group 'default' not found")

sg-0c5f25c14bd16c426


In [92]:
ec2_client = boto3.client('ec2')

# Autorizando o acesso SSH (porta 22) para o security group default
try:
    ec2_client.authorize_security_group_ingress(
        GroupId=security_group_id,
        IpPermissions=[
            {
                'IpProtocol': 'tcp',
                'FromPort': 22,
                'ToPort': 22,
                'IpRanges': [{'CidrIp': '0.0.0.0/0'}]
            }
        ]
    )
except ClientError as e:
    if 'InvalidPermission.Duplicate' in str(e):
        print("SSH access already authorized for security group 'default'")
    else:
        raise

SSH access already authorized for security group 'default'


In [93]:
# Criando um key pair para acessar a instância via SSH
keyname = 'project'
try:
    response = ec2_client.create_key_pair(KeyName=keyname)
except ClientError as e:
    if 'InvalidKeyPair.Duplicate' in str(e):
        print(f"Key pair '{keyname}' already exists. Deleting it and creating a new one.")
        ec2_client.delete_key_pair(KeyName=keyname)
        response = ec2_client.create_key_pair(KeyName=keyname)
    else:
        raise

Key pair 'project' already exists. Deleting it and creating a new one.


In [94]:
# Salvando a chave privada em um arquivo .pem
with open(f'{keyname}.pem', 'w') as file:
    file.write(response['KeyMaterial'])

## Criando instância e volume

In [95]:
def create_instance():
    global security_group_id, keyname
    # Garantindo que eu tenho permissão para criar a instância
    image_id = 'ami-0a59ec92177ec3fad'  # Amazon Linux 2023 AMI 2023.11.20260509.0 x86_64 HVM kernel-6.1

    # Executando um DryRun para garantir que eu poderia executar
    try:
        ec2_resource.create_instances(
            ImageId=image_id,
            MinCount=1,
            MaxCount=1,
            InstanceType='t2.micro',
            DryRun=True,
        )
    except ClientError as e:
        if 'DryRunOperation' not in str(e):
            raise


    instances = ec2_resource.create_instances(
        ImageId=image_id,
        MinCount=1,
        MaxCount=1,
        KeyName=keyname,
        InstanceType='t2.micro',
        DryRun=False,
        NetworkInterfaces=[
            {
                'AssociatePublicIpAddress': True,
                'DeviceIndex': 0,
                'Groups': [security_group_id]
            }
        ]
    )

    return instances[0]

In [96]:
instance = create_instance()
instance

ec2.Instance(id='i-0b24fec87c7831339')

In [97]:
instance.reload()
print(instance.public_ip_address)

None


In [98]:
# Criando o volume EBS
volume = ec2_resource.create_volume(
    AvailabilityZone=instance.placement['AvailabilityZone'],
    Size=8,  # Tamanho do volume em GB
    VolumeType='gp2'  # Tipo do volume (gp2 é o tipo padrão de uso geral)
)

In [99]:
waiter = ec2_client.get_waiter('volume_available')
waiter.wait(VolumeIds=[volume.id])

In [100]:
instance.wait_until_running()
instance.reload()

# Anexando o volume à instância
volume.attach_to_instance(
    InstanceId=instance.id,
    Device='/dev/sdf'  # Dispositivo onde o volume será anexado na instância
)
waiter = ec2_client.get_waiter('volume_in_use')
waiter.wait(VolumeIds=[volume.id])

## Conectando na VM via SSH

In [101]:
HOSTNAME = instance.public_ip_address
USERNAME = 'ec2-user'
KEYPATH = f'{keyname}.pem'

ssh = paramiko.SSHClient()

# Adicionando a chave do host automaticamente (não recomendado para produção)
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

# Conectando
ssh.connect(
    hostname=HOSTNAME,
    username=USERNAME,
    key_filename=KEYPATH
)

print("Conectado!")

stdin, stdout, stderr = ssh.exec_command("uname -a")

# Testando se a conexão está funcionando
print(stdout.read().decode())

Conectado!
Linux ip-172-31-47-141.ec2.internal 6.1.170-210.320.amzn2023.x86_64 #1 SMP PREEMPT_DYNAMIC Fri May  8 17:56:51 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux



### Formatando e configurando o volume

In [102]:
stdin, stdout, stderr = ssh.exec_command("lsblk -f")

print(stdout.read().decode())

NAME      FSTYPE FSVER LABEL UUID                                 FSAVAIL FSUSE% MOUNTPOINTS
xvda                                                                             
├─xvda1   xfs          /     d0446408-7bbd-4991-a80c-ae7c5a8fabdc    6.4G    19% /
├─xvda127                                                                        
└─xvda128 vfat   FAT16       1419-FFEF                               8.7M    13% /boot/efi
xvdf                                                                             



In [103]:
stdin, stdout, stderr = ssh.exec_command(
    "sudo mkfs.ext4 /dev/xvdf",
    get_pty=True
)

print(stderr.read().decode())

In [104]:
stdin, stdout, stderr = ssh.exec_command("lsblk -f")

print(stdout.read().decode())

NAME      FSTYPE FSVER LABEL UUID                                 FSAVAIL FSUSE% MOUNTPOINTS
xvda                                                                             
├─xvda1   xfs          /     d0446408-7bbd-4991-a80c-ae7c5a8fabdc    6.4G    20% /
├─xvda127                                                                        
└─xvda128 vfat   FAT16       1419-FFEF                               8.7M    13% /boot/efi
xvdf      ext4   1.0         4d041949-7f9e-43f5-8b30-4bda8ddb05ca                



In [105]:
stdin, stdout, stderr = ssh.exec_command("sudo mkdir -p /mnt/data")

In [106]:
# Montando o volume EBS
stdin, stdout, stderr = ssh.exec_command(
    "sudo mount /dev/xvdf /mnt/data",
    get_pty=True
)

print(stdout.read().decode())
print(stderr.read().decode())

In [107]:
# Dando permissão de escrita para o usuário ec2-user
stdin, stdout, stderr = ssh.exec_command(
    "sudo chown ec2-user:ec2-user /mnt/data"
)

print(stdout.read().decode())

In [108]:
# Confirmação de que o volume está montado e com permissão de escrita
stdin, stdout, stderr = ssh.exec_command("df -h")

print(stdout.read().decode())

Filesystem      Size  Used Avail Use% Mounted on
devtmpfs        4.0M     0  4.0M   0% /dev
tmpfs           481M     0  481M   0% /dev/shm
tmpfs           193M  448K  192M   1% /run
/dev/xvda1      8.0G  1.6G  6.4G  20% /
tmpfs           481M     0  481M   0% /tmp
/dev/xvda128     10M  1.3M  8.7M  13% /boot/efi
tmpfs            97M     0   97M   0% /run/user/1000
/dev/xvdf       7.8G   24K  7.4G   1% /mnt/data



### Subindo os arquivos na instância e no volume

In [109]:
ROOT = os.getcwd()

# Subindo o arquivo test.txt para a instância tanto local quanto no volume
sftp = ssh.open_sftp()
sftp.put(os.path.join(ROOT, '..', 'data', 'test.txt'), 'test.txt')
sftp.put(os.path.join(ROOT, '..', 'data', 'test.txt'), '/mnt/data/test.txt')

<SFTPAttributes: [ size=71 uid=1000 gid=1000 mode=0o100664 atime=1778773033 mtime=1778773033 ]>

In [110]:
# Mostrando que o arquivo foi enviado
stdin, stdout, stderr = ssh.exec_command("ls -la")
print("STDOUT:")
print(stdout.read().decode())

stdin, stdout, stderr = ssh.exec_command("ls /mnt/data -la")
print("STDOUT:")
print(stdout.read().decode())

STDOUT:
total 16
drwx------. 3 ec2-user ec2-user  90 May 14 15:37 .
drwxr-xr-x. 3 root     root      22 May 14 15:37 ..
-rw-r--r--. 1 ec2-user ec2-user  18 Jan 28  2023 .bash_logout
-rw-r--r--. 1 ec2-user ec2-user 141 Jan 28  2023 .bash_profile
-rw-r--r--. 1 ec2-user ec2-user 492 Jan 28  2023 .bashrc
drwx------. 2 ec2-user ec2-user  29 May 14 15:37 .ssh
-rw-rw-r--. 1 ec2-user ec2-user  71 May 14 15:37 test.txt

STDOUT:
total 24
drwxr-xr-x. 3 ec2-user ec2-user  4096 May 14 15:37 .
drwxr-xr-x. 3 root     root        18 May 14 15:37 ..
drwx------. 2 root     root     16384 May 14 15:37 lost+found
-rw-rw-r--. 1 ec2-user ec2-user    71 May 14 15:37 test.txt



In [111]:
ssh.close()

## Criando nova instância e associando o volume antigo

In [112]:
# Deletando a instância antiga, criando uma nova e associando o volume a ela
instance.terminate()
instance.wait_until_terminated()

instance = create_instance()
instance.wait_until_running()
instance.reload()

volume.attach_to_instance(
    InstanceId=instance.id,
    Device='/dev/sdf'
)
waiter = ec2_client.get_waiter('volume_in_use')
waiter.wait(VolumeIds=[volume.id])

In [113]:
print(instance.public_ip_address)

3.84.202.113


## Conectando SSH na nova instância

In [114]:
HOSTNAME = instance.public_ip_address
USERNAME = 'ec2-user'
KEYPATH = f'{keyname}.pem'

ssh = paramiko.SSHClient()

# Adicionando a chave do host automaticamente (não recomendado para produção)
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

# Conectando
ssh.connect(
    hostname=HOSTNAME,
    username=USERNAME,
    key_filename=KEYPATH
)

print("Conectado!")

stdin, stdout, stderr = ssh.exec_command("uname -a")

# Testando se a conexão está funcionando
print(stdout.read().decode())

Conectado!
Linux ip-172-31-35-253.ec2.internal 6.1.170-210.320.amzn2023.x86_64 #1 SMP PREEMPT_DYNAMIC Fri May  8 17:56:51 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux



In [115]:
# Criando a pasta para montar o volume e montando o volume
stdin, stdout, stderr = ssh.exec_command("sudo mkdir -p /mnt/data")
print("STDOUT:")
print(stdout.read().decode())
print("STDERR:")
print(stderr.read().decode())

stdin, stdout, stderr = ssh.exec_command("sudo mount /dev/sdf /mnt/data")
print("STDOUT:")
print(stdout.read().decode())
print("STDERR:")
print(stderr.read().decode())

STDOUT:

STDERR:

STDOUT:

STDERR:



In [116]:
# Dando permissão pro usuário ec2
stdin, stdout, stderr = ssh.exec_command("sudo chown ec2-user:ec2-user /mnt/data")
print("STDOUT:")
print(stdout.read().decode())
print("STDERR:")
print(stderr.read().decode())

STDOUT:

STDERR:



In [117]:
# Mostrando que o arquivo foi enviado
stdin, stdout, stderr = ssh.exec_command("ls -la")
print("STDOUT:")
print(stdout.read().decode())

stdin, stdout, stderr = ssh.exec_command("ls /mnt/data -la")
print("STDOUT:")
print(stdout.read().decode())

STDOUT:
total 12
drwx------. 3 ec2-user ec2-user  74 May 14 15:38 .
drwxr-xr-x. 3 root     root      22 May 14 15:38 ..
-rw-r--r--. 1 ec2-user ec2-user  18 Jan 28  2023 .bash_logout
-rw-r--r--. 1 ec2-user ec2-user 141 Jan 28  2023 .bash_profile
-rw-r--r--. 1 ec2-user ec2-user 492 Jan 28  2023 .bashrc
drwx------. 2 ec2-user ec2-user  29 May 14 15:38 .ssh

STDOUT:
total 24
drwxr-xr-x. 3 ec2-user ec2-user  4096 May 14 15:37 .
drwxr-xr-x. 3 root     root        18 May 14 15:38 ..
drwx------. 2 root     root     16384 May 14 15:37 lost+found
-rw-rw-r--. 1 ec2-user ec2-user    71 May 14 15:37 test.txt

